<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Requirements Gathering

The laboratory must transform a folder of planar-chessboard images into a calibrated camera model and a reproducible set of numerical and visual validation results.

## Functional Requirements

- discover calibration images from a relative path;
- detect and refine chessboard corners;
- reject invalid views explicitly;
- generate planar calibration coordinates;
- estimate one normalized homography per valid view;
- estimate a shared intrinsic matrix;
- recover one pose per retained view;
- compute reprojection residuals;
- save figures under `outputs/figures/`;
- expose intermediate results required for diagnosis.

## Data Requirements

| Requirement | Specification |
| --- | --- |
| Input | JPEG calibration images |
| Location | `data/calibration_images/` |
| Pattern | 8 × 6 internal corners |
| Square size | 0.03 m |
| Minimum valid views | 3 |
| Geometry unit | metres |
| Error unit | pixels |

## Input / Output Contract

### Inputs
`data/calibration_images/*.jpg`

### Numerical outputs
\(K\), per-view \(R\) and \(t\), projected points, point-wise errors, mean error, and RMSE.

### Saved visual outputs
Detected corners, homography pipeline, camera poses, reprojection overlays, per-view error chart, and residual distribution.

## Assumptions and Constraints

The target is planar, target geometry is known, image ordering is deterministic, paths are repository-relative, at least three views must remain valid, and numerical degeneracy must raise an explicit error.

## Proposed Approach

```text
Calibration images
        ↓
Input validation
        ↓
Corner detection + sub-pixel refinement
        ↓
Planar coordinates
        ↓
Point normalization
        ↓
Normalized DLT homographies
        ↓
Zhang constraint matrix V
        ↓
SVD solution Vb = 0
        ↓
Intrinsic matrix K
        ↓
Pose recovery R, t
        ↓
Reprojection
        ↓
Quantitative + visual evaluation
```

## Pipeline Skeleton

1. Prepare inputs and paths.
2. Build all per-view geometric quantities.
3. Stack intrinsic constraints from valid homographies.
4. Recover the intrinsic matrix.
5. Recover one camera pose per view.
6. Reproject calibration points.
7. Evaluate residuals globally and per view.
8. Save diagnostic figures.
9. Run validation checks.

## Method Selection and Rationale

| Need | Method |
| --- | --- |
| Corner measurement | OpenCV chessboard detection + sub-pixel refinement |
| Plane-to-image mapping | Normalized DLT |
| Homogeneous least-squares systems | SVD |
| Intrinsic calibration | Zhang planar method |
| Rotation cleanup | SVD projection to a proper rotation |
| Model evaluation | Reprojection error |

## Evaluation Metrics

- per-view mean reprojection error;
- per-view reprojection RMSE;
- overall mean reprojection error;
- overall RMSE;
- point-wise residual distribution;
- visual agreement between detected and reprojected points.

## Validation Strategy

1. Validate input paths and files.
2. Validate chessboard detections.
3. Require at least three valid views.
4. Reject non-finite or degenerate homographies.
5. Validate \(K\).
6. Validate rotation orthonormality and determinant.
7. Validate reprojection residuals.
8. Confirm all required output figures exist.

## Failure Modes / Risks

| Failure | Handling |
| --- | --- |
| Missing image directory | `FileNotFoundError` |
| No JPEG inputs | `FileNotFoundError` |
| Corner detection fails | Skip view with message |
| Too few valid views | `RuntimeError` |
| Degenerate point normalization | `ValueError` |
| Degenerate homography / intrinsic system | `ValueError` |
| Improper numerical rotation | SVD correction |

## Implementation Plan

The implementation notebook follows the repository-wide order:

1. Environment and Imports
2. Configuration
3. Paths
4. Core Functions / Classes
5. Data Loading
6. Data Validation
7. Pipeline Implementation
8. Execution
9. Results
10. Quantitative Evaluation
11. Visual Evaluation
12. Save Outputs
13. Validation Checks
14. Final Result Summary

## Expected Deliverables

Four standardized notebooks, unchanged source data and dependencies, reproducible calibration outputs, organized figures, and a concise README.